# Portfolio and Benchmark setup

This notebook demonstrates the creation of portfolios & public indices reconstruction as benchmark and then backtesting them together.

### Objective
We want to reconstruct the **S&P 500** index using float-adjusted market-cap methodology so that we can replicate the SPDR S&P 500 ETF (SPY) and the index level (^GSPC) for the year 2025 using our own backtesting engine. We cross-check the reconstruction against the tradable SPY ETF and the ^GSPC price index, and we go one step further by defining our own investment thesis — a concentrated **Magnificent 8** book — to see whether we can beat the broad market.

All reference data (index constituents + float-adjusted shares outstanding) and all end-of-day prices are sourced from **Refinitiv (LSEG)**.

### Overview

This notebook covers:
- Reconstruction of the S&P 500 index (`.SPX` constituents) via Refinitiv and its daily positional holdings
- Loading float-adjusted shares outstanding from Refinitiv for market-cap weighting
- Creation of an investable SPY ETF portfolio to validate the S&P 500 reconstruction
- Creation of an investable ^GSPC index-level portfolio as a second cross-check
- A concentrated Magnificent 8 thesis portfolio (all S&P 500 constituents)
- Back testing all four portfolios together with market-weighted & equal-weighted approaches

### Prerequisites

Before running this notebook, ensure you have:
- Required Python packages installed (incl. `lseg.data` / `refinitiv.data`)
- A valid Refinitiv LSEG session (the loader opens it automatically)
- Database credentials stored in keyring
- Appropriate database schema and functions
- ipython-sql to run sql in the notebook

## 1. Environment Setup and Path Configuration

Set up the Python environment by configuring import paths and ensuring juypyter notebook can access custom modules in the project structure.

In [1]:
import sys
import os

import pandas as pd

# Get the current notebook's directory and go up to parent
current_dir = os.getcwd()
parent_dir = os.path.dirname(os.path.dirname(current_dir))

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

print(f"Current directory: {current_dir}")
print(f"Added to path: {parent_dir}")
data_eng_path = os.path.join(parent_dir, 'data_engineering')
files_path = os.path.join(parent_dir, 'toolkit\\files')
print(f"Data engineering folder exists: {os.path.exists(data_eng_path)}")
print(f"Files folder exists: {os.path.exists(files_path)}")

Current directory: c:\SourceCode\InvestmentManagement\toolkit\notebooks
Added to path: c:\SourceCode\InvestmentManagement
Data engineering folder exists: True
Files folder exists: True


## 2. Database Connection and Module Imports

Establish database connectivity and import required modules for data engineering and market data retrieval.<br> 
It includes retry logic for database connection failures and sets up SQL magic commands for inline SQL execution.


In [2]:
from data_engineering.database import database

# Open the Refinitiv LSEG desktop session (requires LSEG Workspace running)
import lseg.data as ld
ld.open_session()  # idempotent

# Live DB connection via OS keyring (db/uid/pwd in 'ihub_sql_connection')
engine, connection, conn_str, session = database.get_db_connection()
print('DB connection:', 'OK')

%load_ext sql
%sql $conn_str
from sqlalchemy import text


Database connection successful.
DB connection: OK


In [3]:
from data_engineering.eod_data import refinitiv, yahoo, tiingo

## 3. Backtest Period Definition

Setting the analysis period for the backtest. These dates define the time window for all portfolio performance calculations and risk metrics.

In [4]:
start_date = "2025-01-01"
end_date = "2025-12-31"

In [5]:
%%sql
TRUNCATE TABLE [dbo].[portfolio]
TRUNCATE TABLE [dbo].[portfolio_holdings]
TRUNCATE TABLE [dbo].[market_data]
TRUNCATE TABLE [dbo].[currency_rates]
TRUNCATE TABLE [analytics].[portfolio_analytics]
TRUNCATE TABLE [dbo].[security_fundamentals]

 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
(pyodbc.ProgrammingError) ('42S02', '[42S02] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot find the object "currency_rates" because it does not exist or you do not have permissions. (4701) (SQLExecDirectW)')
[SQL: TRUNCATE TABLE [dbo].[portfolio]
TRUNCATE TABLE [dbo].[portfolio_holdings]
TRUNCATE TABLE [dbo].[market_data]
TRUNCATE TABLE [dbo].[currency_rates]
TRUNCATE TABLE [analytics].[portfolio_analytics]
TRUNCATE TABLE [dbo].[security_fundamentals]]
(Background on this error at: https://sqlalche.me/e/20/f405)


##### Load S&P 500 constituents & float-adjusted shares from Refinitiv
Reconstruct `.SPX` membership for the backtest window, enrich it with our `security_master` identifiers, and write it to `reference.index_constituents` under `index_id = 2`. We also pull float-adjusted shares outstanding from Refinitiv — the source for market-cap weighting later on.

Note: `enrich_with_security_master` hardcodes `index_id = 1`, so we override it to `2` here. The loader now resolves `security_id` via `security_vendor_xref.vendor_ticker` (exact RIC -> normalized RIC -> ISIN/SEDOL/CUSIP/FIGI) and KEEPS `Constituent RIC`, so `build_float_adjusted_shares` correctly receives the enriched frame.

In [6]:
import lseg.data as ld
from data_engineering.index_constituents import refinitiv as ic_ref

SP500_INDEX_ID = 2

index_df = ic_ref.build_index_constituents('.SPX', start_date, end_date)

asset_attributes = ld.get_data(
    universe=index_df['Constituent RIC'].drop_duplicates().tolist(),
    fields=[
        'TR.CommonName', 'TR.ISIN', 'TR.SEDOL', 'TR.CUSIP',
        'TR.ExchangeCountryCode', 'TR.Currency', 'TR.GICSSector',
        'TR.GICSIndustryGroup', 'TR.GICSIndustry', 'TR.ExchangeTicker',
        'TR.ExchangeCode',
    ],
)

joined = index_df.merge(
    asset_attributes, left_on='Constituent RIC', right_on='Instrument', how='left'
)

df_enriched = ic_ref.enrich_with_security_master(joined)
df_enriched['index_id'] = SP500_INDEX_ID  # override hardcoded index_id = 1

# index_constituents requires exchange_ticker, start_date, upsert_by (NOT NULL)
assert not df_enriched['exchange_ticker'].isna().any(), 'exchange_ticker has nulls'
assert not df_enriched['start_date'].isna().any(), 'start_date has nulls'
unresolved = int(df_enriched['security_id'].isna().sum())
print(f"RIC->security_id coverage: {len(df_enriched) - unresolved}/{len(df_enriched)} "
      f"({unresolved} unresolved)")
database.write_index_constituents(df_enriched, session)

# Float-adjusted shares outstanding -> market-cap weighting source.
# Pass the ENRICHED frame (needs Constituent RIC + exchange_ticker + security_id).
metrics_df = ic_ref.build_float_adjusted_shares(df_enriched)
metrics_df['metric_type'] = 'shares_outstanding'  # match the reader's expectation
assert not metrics_df['security_id'].isna().any(), 'shares metric has unresolved security_id'
assert not metrics_df['effective_date'].isna().any(), 'effective_date has nulls'
database.write_security_fundamentals(metrics_df, session)

print(f"Wrote {len(df_enriched)} S&P 500 constituent rows "
      f"(index_id={SP500_INDEX_ID}) and {len(metrics_df)} "
      f"shares-outstanding rows from Refinitiv.")


Database connection successful.
RIC->security_id coverage: 525/525 (0 unresolved)
[shares] pulling float-adjusted shares for 524 instruments over 2025-01-01..2025-12-31 in chunks of 20
  [shares] progress 10/27 chunks done (10 returned data)
  [shares] progress 20/27 chunks done (20 returned data)
  [shares] progress 27/27 chunks done (27 returned data)
  [shares] progress 10/27 chunks done (10 returned data)
  [shares] progress 20/27 chunks done (20 returned data)
  [shares] progress 27/27 chunks done (27 returned data)
Upserting 131341 security_fundamentals rows (non-destructive MERGE)...
Security fundamentals data successfully written (131341 rows upserted).
Wrote 525 S&P 500 constituent rows (index_id=2) and 132590 shares-outstanding rows from Refinitiv.


## 4. Portfolio Creation - Magnificent 8 Tech Stocks

Creating the first portfolio containing concentrated positions in the "Magnificent 8" technology stocks.

In [7]:
result = %sql INSERT INTO [dbo].[portfolio] (portfolio_short_name, portfolio_name, portfolio_type, is_active) \
         OUTPUT INSERTED.port_id \
         VALUES ('MAG8', 'Backtest Mag 8', 'Portfolio', 1);

new_port_id = result[0][0]

 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.


Confirming the Magnificent 8 portfolio was successfully created in the database.

In [8]:
%%sql
SELECT * FROM [dbo].[portfolio] WHERE port_id = :new_port_id

 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.


port_id,portfolio_short_name,portfolio_name,portfolio_type,is_active,reporting_currency,base_currency
1,MAG8,Backtest Mag 8,Portfolio,1,None,None


##### Generate Holdings History for Magnificent 8 Portfolio
Create daily position & holding units for the Magnificent 8 stocks across all business days in the backtest period.<br>
Store the data in portfolio_holdings table<br> 
Verify that the data has been succesfully stored

In [9]:
%%sql
;WITH securities AS (
    SELECT   sm.security_id, x.vendor_ticker AS symbol
    FROM    dbo.security_master sm
    JOIN    dbo.security_vendor_xref x
            ON  x.security_id = sm.security_id
            AND x.vendor = 'Refinitiv'
    WHERE   x.vendor_ticker in ('MSFT.O','NVDA.O','TSLA.O','AMZN.O','AAPL.O','AVGO.O','GOOGL.O','META.O')
),
dates AS (
    SELECT CAST(:start_date AS DATE) dt
    UNION ALL
    SELECT DATEADD(DAY, 1, dt) FROM dates WHERE dt <= :end_date
),
business_dates AS (
    SELECT dt FROM dates WHERE DATEPART(WEEKDAY, dt) BETWEEN 2 AND 6
),
share_quantities AS (
    SELECT    security_id
            , symbol,
               CASE symbol
                   WHEN 'MSFT.O' THEN 5
                   WHEN 'GOOGL.O' THEN 4
                   WHEN 'AVGO.O' THEN 3
                   WHEN 'META.O' THEN 2
                   ELSE 1
               END AS shares
    FROM    securities
)
INSERT INTO dbo.portfolio_holdings(as_of_date, port_id, security_id, held_shares, upsert_date, upsert_by)
SELECT      bd.dt
           ,:new_port_id
           ,sq.security_id
           ,sq.shares AS held_shares
           ,GETDATE() AS upsert_date
           ,SYSTEM_USER AS upsert_by
FROM        business_dates bd
CROSS JOIN  share_quantities sq
ORDER BY    bd.dt, sq.symbol
OPTION (MAXRECURSION 32767);

----------------------------------------------------------------------------------------
SELECT TOP 10 x.vendor_ticker, ph.*
FROM		dbo.portfolio_holdings ph
INNER JOIN	dbo.security_vendor_xref x
			ON  x.security_id = ph.security_id
			AND x.vendor = 'Refinitiv'
WHERE		port_id = :new_port_id
ORDER BY	as_of_date


 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.
2096 rows affected.
Done.


vendor_ticker,ph_id,as_of_date,port_id,security_id,held_shares,upsert_date,upsert_by
AAPL.O,1,2025-01-01,1,7,1.0,Sep 11 2026 9:23AM,MenonPC\menon
AMZN.O,2,2025-01-01,1,13,1.0,Sep 11 2026 9:23AM,MenonPC\menon
AVGO.O,3,2025-01-01,1,6,3.0,Sep 11 2026 9:23AM,MenonPC\menon
GOOGL.O,4,2025-01-01,1,12,4.0,Sep 11 2026 9:23AM,MenonPC\menon
META.O,5,2025-01-01,1,5,2.0,Sep 11 2026 9:23AM,MenonPC\menon
MSFT.O,6,2025-01-01,1,11,5.0,Sep 11 2026 9:23AM,MenonPC\menon
NVDA.O,7,2025-01-01,1,1,1.0,Sep 11 2026 9:23AM,MenonPC\menon
TSLA.O,8,2025-01-01,1,4,1.0,Sep 11 2026 9:23AM,MenonPC\menon
AAPL.O,9,2025-01-02,1,7,1.0,Sep 11 2026 9:23AM,MenonPC\menon
AMZN.O,10,2025-01-02,1,13,1.0,Sep 11 2026 9:23AM,MenonPC\menon


## 5. S&P 500 Index - Reconstruction (Refinitiv)

Create a benchmark called S&P 500 index. This will be used as the market benchmark to compare the performance & risk of the Mag8 portfolio.

In [10]:
result = %sql INSERT INTO [dbo].[portfolio] (portfolio_short_name, portfolio_name, portfolio_type, is_active) \
         OUTPUT INSERTED.port_id \
         VALUES ('SP500', 'S&P 500 Reconstructed', 'Benchmark', 1);

new_port_id = result[0][0]


 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.


verify that the Benchmark was created

In [11]:
%%sql
SELECT * FROM [dbo].[portfolio] WHERE port_id = :new_port_id

 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.


port_id,portfolio_short_name,portfolio_name,portfolio_type,is_active,reporting_currency,base_currency
2,SP500,S&P 500 Reconstructed,Benchmark,1,None,None


##### Generate Holdings History for S&P 500 index
Create daily position & holding units for all S&P 500 stocks across all business days in the backtest period.<br>
held_shares will be set to 1 (equal weighted) for now, but we will compute the weight of the security in Index later on.<br>
Store the data in portfolio_holdings table<br> 
Verify that the data has been succesfully stored

In [12]:
%%sql
;WITH dates AS (
    SELECT CAST(:start_date AS DATE) dt
    UNION ALL
    SELECT DATEADD(DAY, 1, dt) FROM dates WHERE dt<= :end_date
),
business_dates AS (
    SELECT dt
    FROM dates
    WHERE DATEPART(WEEKDAY, dt) BETWEEN 2 AND 6
),
expanded_constituents AS (
    SELECT
        bd.dt AS as_of_date,
        ic.index_id,
        ic.security_id,
        ic.exchange_ticker
    FROM business_dates bd
    JOIN reference.index_constituents ic
      ON bd.dt >= ic.start_date
     AND bd.dt <= ISNULL(ic.end_date, :end_date)
     AND ic.index_id = 2
)
INSERT INTO dbo.portfolio_holdings (as_of_date, port_id, security_id, held_shares, upsert_date, upsert_by)
SELECT  ec.as_of_date,
        :new_port_id,
        ec.security_id,
        1,
        GETDATE(),
        SYSTEM_USER
FROM expanded_constituents ec
OPTION (MAXRECURSION 400);


 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.
131323 rows affected.


[]

In [13]:
database.read_portfolio_holdings(session,engine, start_date, end_date)

all_portfolio_holdings = database.read_portfolio_holdings(session, engine, start_date, end_date)

db_nasdaq_daily_holdings = all_portfolio_holdings[all_portfolio_holdings["port_id"] == new_port_id]
db_nasdaq_daily_holdings 

,as_of_date,port_id,security_id,held_shares
2088,2025-01-01,2,5264,1.0
2089,2025-01-01,2,10785,1.0
2090,2025-01-01,2,174,1.0
2091,2025-01-01,2,10784,1.0
2092,2025-01-01,2,15728,1.0
...,...,...,...,...
133406,2025-12-31,2,13232,1.0
133407,2025-12-31,2,6626,1.0
133408,2025-12-31,2,13978,1.0
133409,2025-12-31,2,10795,1.0


## 6. SPY ETF Portfolio Creation

Construct a portfolio comprising the [SPDR S&P 500 ETF (SPY)](https://www.ssga.com/us/en/intermediary/etfs/funds/spdr-sp-500-etf-trust-spy) to validate the accuracy of our S&P 500 index reconstruction. This ETF will serve as a benchmark, enabling direct comparison between our index reconstruction methodology and the actual market-traded ETF. Ideally, the reconstructed index and the SPY portfolio should exhibit strong alignment—moving in tandem and demonstrating a high degree of correlation.

##### Ensure SPY and ^GSPC exist in security_master
The original Nasdaq demo assumed QQQ/^NDX rows already existed. For S&P 500 we make the ETF and index-level securities explicit so the holdings SQL below resolves.

In [14]:
from sqlalchemy import text

# security_master has NO 'symbol' column; tickers live in security_vendor_xref.vendor_ticker
# (Refinitiv RICs). SPY -> 'SPY.P', ^GSPC -> '.SPX'. Create the master row + xref if missing.
for ric, nm, stype in [
    ('SPY.P',  'SPDR S&P 500 ETF', 'ETF'),
    ('.SPX',   'S&P 500 Index',    'Index'),
]:
    sec_id = session.execute(
        text(
            "SELECT sm.security_id FROM dbo.security_master sm "
            "JOIN dbo.security_vendor_xref x ON x.security_id = sm.security_id "
            "AND x.vendor = 'Refinitiv' WHERE x.vendor_ticker = :r"
        ),
        {'r': ric},
    ).scalar()
    if sec_id is None:
        # insert master row, get its new id, then create the Refinitiv xref row
        res = session.execute(
            text(
                "INSERT INTO dbo.security_master "
                "(name, security_type, asset_class, is_active, upsert_date, upsert_by) "
                "OUTPUT INSERTED.security_id "
                "VALUES (:n, :t, 'Equity', 1, GETDATE(), SYSTEM_USER)"
            ),
            {'n': nm, 't': stype},
        )
        sec_id = res.scalar()
        session.execute(
            text(
                "INSERT INTO dbo.security_vendor_xref "
                "(security_id, vendor, vendor_ticker, is_primary, is_active, upsert_date, upsert_by) "
                "VALUES (:sid, 'Refinitiv', :r, 1, 1, GETDATE(), SYSTEM_USER)"
            ),
            {'sid': sec_id, 'r': ric},
        )
        session.commit()
        print(f'Inserted security_master + xref for {ric} (security_id={sec_id})')
    else:
        print(f'{ric} already present in security_master (security_id={sec_id})')


Inserted security_master + xref for SPY.P (security_id=16890)
Inserted security_master + xref for .SPX (security_id=16891)


In [15]:
result = %sql INSERT INTO [dbo].[portfolio] (portfolio_short_name, portfolio_name, portfolio_type, is_active) \
         OUTPUT INSERTED.port_id \
         VALUES ('SPY', 'S&P 500 ETF', 'Portfolio', 1);

new_port_id = result[0][0]


 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.


Verify that the data has been succesfully stored

In [16]:
%%sql
SELECT * FROM [dbo].[portfolio] WHERE port_id = :new_port_id

 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.


port_id,portfolio_short_name,portfolio_name,portfolio_type,is_active,reporting_currency,base_currency
3,SPY,S&P 500 ETF,Portfolio,1,None,None


##### Generate SPY Holdings History

Create daily holdings for the SPY ETF across the backtest period.<br>Store the data in portfolio_holdings table<br> 
Verify that the data has been succesfully stored

In [17]:
%%sql
;WITH securities AS (
    SELECT   sm.security_id, x.vendor_ticker AS symbol
    FROM     dbo.security_master sm
    JOIN     dbo.security_vendor_xref x
             ON  x.security_id = sm.security_id
             AND x.vendor = 'Refinitiv'
    WHERE    x.vendor_ticker IN ('SPY.P')
),
dates AS (
    SELECT CAST(:start_date AS DATE) dt
    UNION ALL
    SELECT DATEADD(DAY, 1, dt) FROM dates WHERE dt < :end_date
),
business_dates AS (
    SELECT dt FROM dates WHERE DATEPART(WEEKDAY, dt) BETWEEN 2 AND 6
)
INSERT INTO dbo.portfolio_holdings(as_of_date, port_id, security_id, held_shares, upsert_date, upsert_by)
SELECT      bd.dt,
            :new_port_id,
            s.security_id,
            1 AS held_shares,
            GETDATE() AS upsert_date,
            SYSTEM_USER AS upsert_by
FROM        business_dates bd
CROSS JOIN  securities s
ORDER BY    bd.dt, s.security_id
OPTION (MAXRECURSION 32767);

----------------------------------------------------------------------------------------
SELECT TOP 10 x.vendor_ticker, ph.*
FROM		dbo.portfolio_holdings ph
INNER JOIN	dbo.security_vendor_xref x
			ON  x.security_id = ph.security_id
			AND x.vendor = 'Refinitiv'
WHERE		port_id = :new_port_id
ORDER BY	as_of_date


 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.
261 rows affected.
Done.


vendor_ticker,ph_id,as_of_date,port_id,security_id,held_shares,upsert_date,upsert_by
SPY.P,133420,2025-01-01,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133421,2025-01-02,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133422,2025-01-03,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133423,2025-01-06,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133424,2025-01-07,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133425,2025-01-08,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133426,2025-01-09,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133427,2025-01-10,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133428,2025-01-13,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon
SPY.P,133429,2025-01-14,3,16890,1.0,Sep 11 2026 9:23AM,MenonPC\menon


In [18]:
result = %sql INSERT INTO [dbo].[portfolio] (portfolio_short_name, portfolio_name, portfolio_type, is_active) \
         OUTPUT INSERTED.port_id \
         VALUES ('GSPC', 'S&P 500 Price Index', 'Portfolio', 1);

new_port_id = result[0][0]


 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.


In [19]:
%%sql
;WITH securities AS (
    SELECT   sm.security_id, x.vendor_ticker AS symbol
    FROM     dbo.security_master sm
    JOIN     dbo.security_vendor_xref x
             ON  x.security_id = sm.security_id
             AND x.vendor = 'Refinitiv'
    WHERE    x.vendor_ticker IN ('.SPX')
),
dates AS (
    SELECT CAST(:start_date AS DATE) dt
    UNION ALL
    SELECT DATEADD(DAY, 1, dt) FROM dates WHERE dt < :end_date
),
business_dates AS (
    SELECT dt FROM dates WHERE DATEPART(WEEKDAY, dt) BETWEEN 2 AND 6
)
INSERT INTO dbo.portfolio_holdings(as_of_date, port_id, security_id, held_shares, upsert_date, upsert_by)
SELECT      bd.dt,
            :new_port_id,
            s.security_id,
            1 AS held_shares,
            GETDATE() AS upsert_date,
            SYSTEM_USER AS upsert_by
FROM        business_dates bd
CROSS JOIN  securities s
ORDER BY    bd.dt, s.security_id
OPTION (MAXRECURSION 32767);

----------------------------------------------------------------------------------------
SELECT TOP 10 x.vendor_ticker, ph.*
FROM		dbo.portfolio_holdings ph
INNER JOIN	dbo.security_vendor_xref x
			ON  x.security_id = ph.security_id
			AND x.vendor = 'Refinitiv'
WHERE		port_id = :new_port_id
ORDER BY	as_of_date


 * mssql+pyodbc://MENONPC\SQLEXPRESS/ihub?Encrypt=no&TrustServerCertificate=yes&autocommit=true&driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes
Done.
261 rows affected.
Done.


vendor_ticker,ph_id,as_of_date,port_id,security_id,held_shares,upsert_date,upsert_by
.SPX,133681,2025-01-01,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133682,2025-01-02,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133683,2025-01-03,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133684,2025-01-06,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133685,2025-01-07,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133686,2025-01-08,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133687,2025-01-09,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133688,2025-01-10,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133689,2025-01-13,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon
.SPX,133690,2025-01-14,4,16891,1.0,Sep 11 2026 9:23AM,MenonPC\menon


## 7. Market Data Load

Now that we have constructed all our portfolios, it's time to fetch market data for all the securities marked as is_active in the security master.

In [20]:
# df_enriched carries 'exchange_ticker' in TICKER form (e.g. 'CCI', 'MSFT'), which is
# exactly what refinitiv.get_stock_price expects (it converts ticker -> RIC internally).
# security_master has no 'symbol' column, so we source tickers from the enriched frame.
sp500_symbols = list(df_enriched['exchange_ticker'].dropna().unique())
mag8_symbols = ['MSFT', 'NVDA', 'TSLA', 'AMZN', 'AAPL', 'AVGO', 'GOOGL', 'META']
bench_symbols = ['SPY', '^GSPC']
needed_symbols = list(dict.fromkeys(sp500_symbols + mag8_symbols + bench_symbols))

# Build the (symbol, security_id) frame for the EOD loader.
# S&P 500 + Mag8 ids come from df_enriched; bench ids come from security_vendor_xref.
xref = database.read_security_vendor_xref(session, engine, vendor='Refinitiv')
bench_ids = dict(zip(xref['vendor_ticker'], xref['security_id']))
df_sp500 = df_enriched[['exchange_ticker', 'security_id']].rename(columns={'exchange_ticker': 'symbol'})
df_bench = pd.DataFrame({
    'symbol': bench_symbols,
    'security_id': [bench_ids.get(s) for s in bench_symbols],
}).dropna(subset=['security_id'])
df_securities = pd.concat([df_sp500, df_bench], ignore_index=True)
df_securities = df_securities[df_securities['symbol'].isin(needed_symbols)].drop_duplicates('symbol')

data_source = "refinitiv"

if data_source.lower() == 'yahoo':
    df_eod, df_no_eod = yahoo.get_stock_price(
        df_securities[['symbol', 'security_id']], start_date, end_date
    )
else:
    df_eod = refinitiv.get_stock_price(
        df_securities[['symbol', 'security_id']], start_date, end_date
    )

database.write_market_data(df_eod, session)


No RIC found for symbol: 


c:\SourceCode\InvestmentManagement\data_engineering\eod_data\refinitiv.py:161:UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.


Database error: (pyodbc.IntegrityError) ('23000', "[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot insert the value NULL into column 'open', table 'ihub.dbo.market_data'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)")
[SQL: INSERT INTO dbo.market_data (as_of_date, security_id, dividends, stock_splits, interval, dataload_date, price_currency) OUTPUT inserted.md_id VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: (NaT, 174, 0, 0, '1d', '2026-09-11 09:23:45', 'USD')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [21]:
# df_enriched carries 'exchange_ticker' in TICKER form (e.g. 'CCI', 'MSFT').
# refinitiv.get_stock_price expects TICKER-form 'symbol' (it converts ticker -> RIC internally).
# security_master has no 'symbol' column, so we source tickers from the enriched frame and
# resolve each ticker -> Refinitiv RIC -> security_id via security_vendor_xref.
import threading

sp500_symbols = list(df_enriched['exchange_ticker'].dropna().unique())
mag8_symbols = ['MSFT', 'NVDA', 'TSLA', 'AMZN', 'AAPL', 'AVGO', 'GOOG', 'META']
needed_symbols = list(dict.fromkeys(sp500_symbols + mag8_symbols + ['SPY']))

xref = database.read_security_vendor_xref(session, engine, vendor='Refinitiv')

# xref stores RICs (e.g. 'MSFT.O', 'SPY.P', '.SPX'); resolve_tickers_to_rics yields a base RIC
# (e.g. 'SPY'). Build a map keyed by BOTH the raw RIC and a normalized (qualifier-stripped) form
# so 'SPY.P' -> 'SPY' resolves correctly to the same security_id.
def _norm(ric):
    if ric and len(ric) > 2 and ric[-2] == '.' and ric[-1].isalpha():
        return ric[:-2]
    return ric

ric_to_secid = {}
for vt, sid in zip(xref['vendor_ticker'], xref['security_id']):
    ric_to_secid[vt] = sid
    ric_to_secid[_norm(vt)] = sid

# Resolve tickers -> Refinitiv RICs -> security_id (only symbols with a valid xref mapping kept)
ric_df = refinitiv.resolve_tickers_to_rics(pd.DataFrame({'symbol': needed_symbols}))
ric_df['security_id'] = ric_df['ric'].map(ric_to_secid)
df_securities = ric_df.dropna(subset=['security_id'])[['symbol', 'security_id']].copy()
print(f"EOD universe resolved: {len(df_securities)} securities (from {len(needed_symbols)} requested)")

data_source = "refinitiv"

def _fetch_batch(df_b, timeout=900):
    """Fetch one batch in a worker thread so a TRULY hung Refinitiv call can't stall the run."""
    out = {}
    def _run():
        out['df'] = refinitiv.get_stock_price(df_b, start_date, end_date)
    t = threading.Thread(target=_run, daemon=True)
    t.start(); t.join(timeout)
    if t.is_alive():
        return None, "TIMEOUT"
    return out.get('df'), "ok"

BATCH = 50
rows = [r for _, r in df_securities[['symbol', 'security_id']].iterrows()]
parts = [rows[i:i+BATCH] for i in range(0, len(rows), BATCH)]
print(f"EOD pull: {len(rows)} symbols in {len(parts)} batches of {BATCH}")

total_rows = 0
skipped = 0
for b, batch in enumerate(parts, 1):
    df_b = pd.DataFrame(batch)
    try:
        df_out, status = _fetch_batch(df_b)
    except Exception as e:
        df_out, status = None, f"ERR {type(e).__name__}: {str(e)[:80]}"
    if df_out is not None and not df_out.empty:
        database.write_market_data(df_out, session)
        total_rows += len(df_out)
        print(f"  batch {b}/{len(parts)} OK ({len(df_out)} rows, running total {total_rows}) [{status}]")
    else:
        skipped += len(df_b)
        print(f"  batch {b}/{len(parts)} SKIPPED ({len(df_b)} symbols) [{status}]")

# S&P 500 INDEX (.SPX) does not resolve via ticker conversion (^GSPC -> NA); fetch its RIC directly.
gspc_sec_id = ric_to_secid.get('.SPX')
if gspc_sec_id is not None:
    raw = refinitiv._fetch_refinitiv_data(['.SPX'], start_date, end_date, 'D')
    if not raw.empty:
        idx_valid = pd.DataFrame({'security_id': [gspc_sec_id], 'ric': ['.SPX'], 'symbol': ['^GSPC']})
        df_gspc = refinitiv._standardize_dataframe(raw, idx_valid, pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'), '1d')
        database.write_market_data(df_gspc, session)
        total_rows += len(df_gspc)
        print(f"  GSPC (.SPX) fetched: {len(df_gspc)} rows")
    else:
        print("  GSPC (.SPX) fetch returned no data")

print(f"EOD pull complete: {total_rows} rows written, {skipped} symbols skipped")


EOD universe resolved: 507 securities (from 517 requested)
EOD pull: 507 symbols in 11 batches of 50


c:\SourceCode\InvestmentManagement\data_engineering\eod_data\refinitiv.py:161:UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.


Database error: (pyodbc.IntegrityError) ('23000', "[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot insert the value NULL into column 'open', table 'ihub.dbo.market_data'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)")
[SQL: INSERT INTO dbo.market_data (as_of_date, security_id, dividends, stock_splits, interval, dataload_date, price_currency) OUTPUT inserted.md_id VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: (NaT, 174.0, 0, 0, '1d', '2026-09-11 09:25:22', 'USD')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
  batch 1/11 OK (6773 rows, running total 6773) [ok]


c:\SourceCode\InvestmentManagement\data_engineering\eod_data\refinitiv.py:161:UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.


Database error: (pyodbc.IntegrityError) ('23000', "[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot insert the value NULL into column 'volume', table 'ihub.dbo.market_data'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)")
[SQL: INSERT INTO dbo.market_data (as_of_date, security_id, [open], high, low, [close], adj_close, dividends, stock_splits, interval, dataload_date, price_currency) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: (Timestamp('2025-01-03 00:00:00'), 1.0, 622000.0, 629000.0, 622000.0, 626740.0, 626740.0, 0, 0, '1d', '2026-09-11 09:25:57', 'USD', Timestamp('2025-01-03 00:00:00'), 1.0, 622000.0, 629000.0, 622000.0, 626740.0, 626740.0, 0, 0, '1d', '2026-09-11 09:25:57', 'USD')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
  batch 2/11 OK (6870 rows, running total 13643

...

Database error: (pyodbc.IntegrityError) ('23000', "[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot insert the value NULL into column 'open', table 'ihub.dbo.market_data'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)")
[SQL: INSERT INTO dbo.market_data (as_of_date, security_id, dividends, stock_splits, interval, dataload_date, price_currency) VALUES (?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?)]
[parameters: (NaT, 130.0, 0, 0, '1d', '2026-09-11 09:27:41', 'USD', NaT, 147.0, 0, 0, '1d', '2026-09-11 09:27:41', 'USD')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
  batch 9/11 OK (7271 rows, running total 67528) [ok]


c:\SourceCode\InvestmentManagement\data_engineering\eod_data\refinitiv.py:161:UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.


Database error: (pyodbc.IntegrityError) ('23000', "[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot insert the value NULL into column 'open', table 'ihub.dbo.market_data'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)")
[SQL: INSERT INTO dbo.market_data (as_of_date, security_id, dividends, stock_splits, interval, dataload_date, price_currency) OUTPUT inserted.md_id VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: (NaT, 27.0, 0, 0, '1d', '2026-09-11 09:27:48', 'USD')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
  batch 10/11 OK (8018 rows, running total 75546) [ok]


c:\SourceCode\InvestmentManagement\data_engineering\eod_data\refinitiv.py:161:UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.


Database error: (pyodbc.IntegrityError) ('23000', "[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot insert the value NULL into column 'open', table 'ihub.dbo.market_data'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)")
[SQL: INSERT INTO dbo.market_data (as_of_date, security_id, dividends, stock_splits, interval, dataload_date, price_currency) VALUES (?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?)]
[parameters: (NaT, 223.0, 0, 0, '1d', '2026-09-11 09:27:55', 'USD', NaT, 239.0, 0, 0, '1d', '2026-09-11 09:27:55', 'USD')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
  batch 11/11 OK (1003 rows, running total 76549) [ok]
  GSPC (.SPX) fetched: 261 rows
EOD pull complete: 76810 rows written, 0 symbols skipped


We also need float-adjusted shares outstanding for all S&P 500 securities to reconstruct the index. Since the S&P 500 is market-cap weighted, we must weight the holdings by market capitalization, computed using the formula below:

<b><i>Market Capitalization = Price × Shares Outstanding</i></b>

The Refinitiv LSEG API provides float-adjusted shares outstanding, which we already loaded into the security_fundamentals table via the `build_float_adjusted_shares` step in Section 5.

In [22]:
# Float-adjusted shares outstanding were already loaded from Refinitiv in the
# 'Load S&P 500 constituents' step (build_float_adjusted_shares ->
# write_security_fundamentals). Verify they landed.
database.read_security_fundamentals(session, engine, 'shares_outstanding').head()
print('Shares outstanding (market-cap weighting source) is ready for the S&P 500 backtest.')


Shares outstanding (market-cap weighting source) is ready for the S&P 500 backtest.


We observe that SPY and the reconstructed S&P 500 move closely together.

In [23]:
from analytics.performance import performance_analytics as perf

portfolio_short_names = ["SPY","SP500","MAG8","GSPC"]
portfolio_market_data = database.get_portfolio_market_data(session, engine, start_date, end_date, portfolio_short_names)
if data_source.lower() == "yahoo":
    portfolio_asset_returns = perf.calculate_portfolio_constituent_returns(portfolio_market_data, "adj_close")["log_returns"]
else:
    portfolio_asset_returns = perf.calculate_portfolio_constituent_returns(portfolio_market_data, "adj_close")["log_returns"].shift(1)
portfolio_asset_weights = perf.calculate_portfolio_constituent_weights(portfolio_market_data, "adj_close", "market_weighted")
portfolio_return = (portfolio_asset_returns * portfolio_asset_weights).T.groupby(level=0).sum().T

portfolio_total_return = portfolio_asset_returns * portfolio_asset_weights
perf.plot_cumulative_returns(portfolio_total_return.T.groupby(level=0).sum().T)


ValueError: No objects to concatenate

#### Equal Weighted

Now lets calculate portfolio returns and weights using the equal weighting methodology.<br>

Understanding Equal Weighting vs. Market Cap</br>
In a market-cap-weighted index (like the standard S&P 500), a 1% move in Apple or Microsoft affects the index much more than a 1% move in a smaller company. In an equal-weighted index, every company has the same impact.

| Feature        | Market-Cap Weighted                          | Equal-Weighted                              |
|---------------|----------------------------------------------|---------------------------------------------|
| Risk Profile  | Concentrated in Mega-Caps                    | More diversified across mid/large caps      |
| Growth Driver | Driven by "Winners" (e.g., MAG8)             | Driven by broad market breadth              |
| Volatility    | Lower (usually)                              | Higher (if smaller caps are volatile)       |




In [ ]:
portfolio_short_names = ["SPY","SP500","MAG8","GSPC"]
portfolio_market_data = database.get_portfolio_market_data(session, engine, start_date, end_date, portfolio_short_names)
portfolio_asset_returns = perf.calculate_portfolio_constituent_returns(portfolio_market_data, "adj_close")["returns"]
portfolio_asset_weights = perf.calculate_portfolio_constituent_weights(portfolio_market_data, "adj_close", "equal_weighted")
portfolio_return = (portfolio_asset_returns * portfolio_asset_weights).T.groupby(level=0).sum().T

portfolio_total_return = portfolio_asset_returns * portfolio_asset_weights
perf.plot_cumulative_returns(portfolio_total_return.T.groupby(level=0).sum().T)


If you had held an equal-weighted version of the S&P 500 during this period

#### Mixed Weighted

Calculate portfolio returns and weights for MAG8 using equal weighting and for SP500 and SPY using market weighting. This approach provides a hybrid perspective on portfolio performance.

In [ ]:
portfolio_short_names = ["SPY","SP500","MAG8","GSPC"]
portfolio_market_data = database.get_portfolio_market_data(session, engine, start_date, end_date, portfolio_short_names)
portfolio_asset_returns = perf.calculate_portfolio_constituent_returns(portfolio_market_data, "adj_close")["returns"]
portfolio_asset_weights = perf.calculate_portfolio_constituent_weights(portfolio_market_data, "adj_close", "market_weighted", portfolio_specific_weights={
        "MAG8": "equal_weighted"
    })
portfolio_return = (portfolio_asset_returns * portfolio_asset_weights).T.groupby(level=0).sum().T

portfolio_total_return = portfolio_asset_returns * portfolio_asset_weights
perf.plot_cumulative_returns(portfolio_total_return.T.groupby(level=0).sum().T)


## 8. Portfolio Risk Analysis

The Mag8 portfolio shows the highest risk, with both historical and parametric VaR values significantly more negative than SPY and the reconstructed S&P 500. SPY sits in the middle, suggesting more concentration risk than the broader S&P 500, which has the lowest risk due to greater diversification. Parametric VaR is consistently more conservative than historical VaR across all three, possibly reflecting volatility clustering or heavier tails in return distributions. This difference is especially noticeable in Mag8, where parametric VaR nearly doubles historical VaR.

In [ ]:
# Calculate Portfolio Risk
from analytics.risk import risk_analytics as risk
max_date_index = portfolio_asset_weights.index.max()
df_portfolio_var = []

for PortfolioShortName in portfolio_short_names:
    portfolio_returns = portfolio_asset_returns[PortfolioShortName]
    portfolio_weights = portfolio_asset_weights[PortfolioShortName]
    portfolio_latest_weights = portfolio_asset_weights[max_date_index:max_date_index][PortfolioShortName]

    obj_risk = risk.PortfolioVaR(portfolio_asset_returns,
                                 portfolio_latest_weights,
                                 PortfolioShortName,
                                 lookback_days=24,
                                 horizon_days=1,
                                 confidence_interval=0.99)

    var_result = obj_risk.calculate_var()
    df_var = pd.DataFrame.from_dict(var_result, orient="columns")
    df_portfolio_var.append(df_var)

df_combined_var = pd.concat(df_portfolio_var, keys=portfolio_short_names, ignore_index=False)
df_combined_var